# 11 — Hardware-Aware QEC with Quantinuum Guppy

## Read before you begin

- [Quantinuum Guppy documentation](https://docs.quantinuum.com/guppy/), especially the **Getting Started** and **Emulator** pages.
- Concepts to extract: Guppy is a Python-embedded *circuit-authoring* language — every instruction (`h`, `cx`, `x`, `measure`, ...) it exposes maps to an actual gate that would run on hardware, unlike Stim, which simulates stabilizer circuits abstractly and never has to worry about how a gate is physically realized. Guppy programs are compiled and can be run locally in a full quantum emulator (`Selene`, bundled with `guppylang`) with no hardware account required — only *submitting* to real Quantinuum hardware needs one.

**External setup:** this notebook uses `guppylang` (PyPI name for Guppy), listed in this repository's `pyproject.toml`; `uv sync` installs it. Every runnable cell in this notebook uses only the local emulator — no account or network access is required. One cell, clearly marked, describes (but does not run) what real-hardware submission looks like.

## Learning objectives

Every notebook so far modeled circuits abstractly, through Stim (a stabilizer simulator, fast because it never represents a real physical gate set) or by hand (Pauli matrices). Guppy is different: it is the kind of language you would actually write to run a QEC circuit on real hardware. By the end of this notebook you should be able to:

1. write and locally emulate a basic Guppy program;
2. explain why Guppy's gate set matters in a way Stim's abstract simulation doesn't — every instruction is a real, physically executable operation;
3. implement the classical repetition code's full workflow (notebooks 02/07's encode → inject error → syndrome measurement → classically-conditioned correction → readout) as one genuine, runnable Guppy program;
4. state clearly what does and doesn't require a hardware account, and see that boundary drawn explicitly in the code.

**Expected outputs:** a locally emulated Bell pair, and a locally emulated 3-qubit bit-flip-code correction that recovers the correct logical value despite an injected fault — the same logical result as notebook 02, now expressed as an actual runnable quantum program instead of classical bit arithmetic.

**Assumes:** notebook 02 (the 3-qubit repetition code, majority-vote decoding) and notebook 01 (ancilla-based syndrome measurement). This is the final notebook in the roadmap.

## 1. A minimal Guppy program

A Guppy function is an ordinary Python function decorated with `@guppy`. Qubits are created with `qubit()`, gates are plain function calls (`h`, `cx`, `x`, ...), and `measure` consumes a qubit and returns a classical bit. `result(name, value)` records a named output for each shot — this is how a Guppy program reports results, since a `@guppy`-decorated function itself has no ordinary Python return value once compiled for the emulator.

In [1]:
from guppylang import guppy
from guppylang.std.builtins import result
from guppylang.std.quantum import cx, h, measure, qubit


@guppy
def bell_pair() -> None:
    q1, q2 = qubit(), qubit()
    h(q1)
    cx(q1, q2)
    result("q1", measure(q1))
    result("q2", measure(q2))


bell_result = bell_pair.emulator(n_qubits=2).with_shots(10).run()
print(bell_result)

EmulatorResult(results=[QsysShot(entries=[('q1', 1), ('q2', 1)]), QsysShot(entries=[('q1', 0), ('q2', 0)]), QsysShot(entries=[('q1', 1), ('q2', 1)]), QsysShot(entries=[('q1', 0), ('q2', 0)]), QsysShot(entries=[('q1', 1), ('q2', 1)]), QsysShot(entries=[('q1', 1), ('q2', 1)]), QsysShot(entries=[('q1', 0), ('q2', 0)]), QsysShot(entries=[('q1', 0), ('q2', 0)]), QsysShot(entries=[('q1', 1), ('q2', 1)]), QsysShot(entries=[('q1', 0), ('q2', 0)])])


### Prediction 1

**Before re-reading the output above:** a Bell pair entangles two qubits so their measurement outcomes always agree. Scan the 10 shots — do `q1` and `q2` ever disagree within a single shot? Would you expect them to?

## 2. Why the gate set matters here (and didn't in Stim)

Every notebook using Stim (01, 03, 05, 08, 09, 10) worked with an abstract stabilizer simulation — `H`, `CX`, `M`, noise channels — chosen for simulation speed, not because those are literally the operations a real device performs. A Guppy program, by contrast, is exactly the sequence of operations that *would* run on hardware (after Quantinuum's own compiler stack maps it to that device's specific native gates — a step this notebook doesn't need to perform by hand, but which is why Guppy programs, unlike Stim circuits, are meant to be submitted to real machines). This is the practical meaning of "hardware-aware": the code you write here is a real program, not a simulator input format.

## 3. The repetition code, as an actual runnable program

Recall notebook 02's `encode`, `flip_bit`, `syndrome`, and `decode` — all plain Python bit arithmetic. Here is the same logical workflow (encode a logical $1$, inject a bit-flip fault, extract a 2-bit syndrome via ancillas exactly like notebook 01, and apply a syndrome-conditioned correction) as one genuine quantum program.

In [2]:
from guppylang.std.quantum import x


@guppy
def bitflip_code_demo() -> None:
    # Encode a logical |1> across 3 data qubits: notebook 02's encode(1) = (1, 1, 1).
    q0, q1, q2 = qubit(), qubit(), qubit()
    x(q0)
    cx(q0, q1)
    cx(q0, q2)

    # Simulate a fault: flip qubit 1 (notebook 02's flip_bit(codeword, 1)).
    x(q1)

    # Syndrome extraction: 2 ancillas, exactly notebook 01's ancilla-parity technique,
    # implementing notebook 02's syndrome() = (b0^b1, b1^b2).
    a0, a1 = qubit(), qubit()
    cx(q0, a0)
    cx(q1, a0)
    cx(q1, a1)
    cx(q2, a1)
    s0 = measure(a0)
    s1 = measure(a1)
    result("s0", s0)
    result("s1", s1)

    # Syndrome-conditioned correction: notebook 02's SYNDROME_TABLE, as real-time control flow.
    if s0 and not s1:
        x(q0)
    elif s0 and s1:
        x(q1)
    elif s1 and not s0:
        x(q2)

    # Final readout: all three should agree (recovered logical 1), despite the injected fault.
    result("q0_final", measure(q0))
    result("q1_final", measure(q1))
    result("q2_final", measure(q2))


demo_result = bitflip_code_demo.emulator(n_qubits=5).with_shots(20).run()
print(demo_result)

EmulatorResult(results=[QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_final', 1), ('q1_final', 1), ('q2_final', 1)]), QsysShot(entries=[('s0', 1), ('s1', 1), ('q0_f

### Prediction 2

**Before running the next cell:** notebook 02's `SYNDROME_TABLE` said syndrome `(1, 1)` means qubit 1 flipped. Look at the `s0`/`s1` values printed above — do they match that, and do all three final readouts agree with each other despite the injected fault on `q1`?

In [3]:
# Check every shot's syndrome and final agreement programmatically, rather than reading by eye.
def shot_is_correct(shot):
    values = dict(shot.entries)
    syndrome_matches_table = (values["s0"], values["s1"]) == (1, 1)
    all_three_agree = values["q0_final"] == values["q1_final"] == values["q2_final"] == 1
    return syndrome_matches_table and all_three_agree


all_correct = all(shot_is_correct(shot) for shot in demo_result.results)
print("Every shot: syndrome (1,1) as notebook 02 predicts, and full logical recovery:", all_correct)

Every shot: syndrome (1,1) as notebook 02 predicts, and full logical recovery: True


Every shot reproduces notebook 02's syndrome table exactly, and every shot recovers the correct logical value after correction — the exact same logical guarantee notebook 02 proved with plain Python bit arithmetic, now demonstrated by an actual runnable quantum program with real qubits, real gates, and real mid-circuit measurement-conditioned control flow.

## 4. What this notebook does *not* do, and why

This program injects one fault deterministically (a single `x(q1)`), the same way notebook 05 injected single, deterministic errors before turning to statistics. Studying *statistical* logical failure rates across many random noise realizations, the way notebook 08 and notebook 10 did, is a job for Stim (and Stim + PyMatching), not Guppy — Stim's abstract stabilizer simulation can sample millions of shots efficiently precisely because it *doesn't* model real gates. Guppy's job is the opposite: express the exact program that would run on real hardware. Large-scale noise statistics and hardware-aware program authoring are complementary, not competing, tools — this is why the roadmap uses Stim for notebooks 05–10 and Guppy only here, at the end.

## 5. Optional: what real-hardware submission looks like (not run here)

Everything above ran on the local `Selene` emulator bundled with `guppylang` — no account, no network access. Submitting the *same* compiled program to actual Quantinuum hardware instead follows a different, account-gated path through Guppy's hardware backends. The cell below is commented out and **not executed** by this notebook; it exists only to show where the boundary between "runs anywhere" and "needs a Quantinuum account" falls.

In [4]:
# OPTIONAL -- requires a Quantinuum account and network access. Not run as part of this notebook.
#
# from guppylang.std.quantum import qubit
#
# job = bitflip_code_demo.compile()          # same compiled program as above
# # Submission to real hardware goes through Quantinuum's separate hardware-access
# # tooling (account credentials, a target machine name, and queueing), which is
# # intentionally outside the scope of this local, reading-led notebook.
# # See https://docs.quantinuum.com/guppy/ for the current hardware-submission guide.
print("This cell is illustrative only -- see the comment above. Nothing was submitted to hardware.")

This cell is illustrative only -- see the comment above. Nothing was submitted to hardware.


## Roadmap complete

This closes the reading-led sequence: classical linear codes → the stabilizer/CSS/Steane trilogy → quantum parity-check matrices and decoding → noisy Steane-code decoding → the surface code's foundations, detectors, and decoding with Stim and Loom → and, here, a real hardware-aware program expressing exactly the kind of circuit those earlier notebooks only ever simulated.

## Exercises

**1.** Section 3's `bitflip_code_demo` always injects the fault on `q1`. Write `bitflip_code_demo_q2()` (a second, separate `@guppy` function — a Guppy function's body is fixed once defined) that instead injects the fault on `q2`, and confirm the printed syndrome now matches notebook 02's `SYNDROME_TABLE` entry for a qubit-2 flip instead.

**2.** Section 1's `bell_pair` never disagreed across `q1`/`q2` because there was no noise anywhere in the circuit. Add an `x` gate on `q2` *before* the `cx` in a copy of `bell_pair`, called `broken_bell_pair`, emulate it for a few shots, and explain in a comment what relationship between `q1` and `q2` you now expect instead, and why the `cx` no longer produces the same correlation.

In [5]:
# TODO: define bitflip_code_demo_q2() here as a full @guppy function (copy Section 3's
# structure and move the injected fault from q1 to q2), then run and inspect it, e.g.:
# result_q2 = bitflip_code_demo_q2.emulator(n_qubits=5).with_shots(20).run()
# print(result_q2)

# TODO: define broken_bell_pair() here, then run and inspect it, e.g.:
# broken_result = broken_bell_pair.emulator(n_qubits=2).with_shots(10).run()
# print(broken_result)